In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU is", "available" if tf.config.list_physical_devices('GPU') else "not available")

TensorFlow version: 2.16.2
GPU is not available


In [ ]:
import pandas as pd

# Charger le fichier Parquet dans un DataFrame
df = pd.read_parquet('data/CICDDOS2019.parquet')

      Protocol  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0           17             48                  2                       0   
1           17              2                  2                       0   
2           17              1                  2                       0   
3           17              1                  2                       0   
4           17              1                  2                       0   
...        ...            ...                ...                     ...   
6698         0       19029778                 12                       0   
6699         6         171239                  1                       1   
6700         6            221                  1                       2   
6701         6          62253                  4                       2   
6702         0       19059205                  9                       0   

      Fwd Packets Length Total  Bwd Packets Length Total  \
0                       294

In [41]:
print("Aperçu des données :", df.head())
print("Résumé des colonnes :", df.info())
print("Label unique :", df["Label"].unique())

Aperçu des données :    Protocol  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0        17             48                  2                       0   
1        17              2                  2                       0   
2        17              1                  2                       0   
3        17              1                  2                       0   
4        17              1                  2                       0   

   Fwd Packets Length Total  Bwd Packets Length Total  Fwd Packet Length Max  \
0                    2944.0                       0.0                 1472.0   
1                    2944.0                       0.0                 1472.0   
2                    2944.0                       0.0                 1472.0   
3                    2944.0                       0.0                 1472.0   
4                    2896.0                       0.0                 1448.0   

   Fwd Packet Length Min  Fwd Packet Length Mean  Fwd Packe

In [34]:
colnames = df.columns.tolist()
colnames

['Protocol',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Fwd Packets Length Total',
 'Bwd Packets Length Total',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Packet Length Min',
 'Packet Length Max',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'PSH Flag Count',
 'ACK Flag Count',
 'URG Flag Count',

In [40]:
df[df.columns[-1:]].dtypes

Label    category
dtype: object

In [36]:
non_numeric_columns = df.select_dtypes(exclude=['int8', 'int16', 'int32', 'int64', 'float32', 'float64']).columns
print("Colonnes non numériques :")
print(non_numeric_columns)

Colonnes non numériques :
Index(['Label'], dtype='object')


In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Séparer les features (X) et les labels (y)
X = df.drop(columns=["Label"]).astype(float).values  # Enlever la colonne 'Label'
y = pd.get_dummies(df["Label"])  # Extraire la colonne 'Label'

In [55]:
# Normaliser les features (important pour les CNN)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshaper les données pour le CNN (le CNN 1D attend un format (samples, timesteps, features))
X = X.reshape(X.shape[0], X.shape[1], 1)  # (6703, 78, 1)

# Diviser les données en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [59]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout

# Construire le modèle CNN
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(y.shape[1], activation='softmax')  # Nombre de neurones = Nombre de classes
])

# Compiler le modèle
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


/Users/childeric/Documents/studies/idia/PRED/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [60]:
# Entraîner le modèle
history = model.fit(
    X_train, y_train, 
    epochs=10,  # Nombre d'époques
    batch_size=32,  # Taille des mini-lots
    validation_split=0.2,  # Fraction des données de validation
    verbose=1  # Afficher les logs d'entraînement
)


Epoch 1/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9285 - loss: 0.2564 - val_accuracy: 0.9972 - val_loss: 0.0322
Epoch 2/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9937 - loss: 0.0354 - val_accuracy: 0.9972 - val_loss: 0.0315
Epoch 3/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9961 - loss: 0.0230 - val_accuracy: 0.9972 - val_loss: 0.0280
Epoch 4/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9952 - loss: 0.0267 - val_accuracy: 0.9972 - val_loss: 0.0249
Epoch 5/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9953 - loss: 0.0211 - val_accuracy: 0.9972 - val_loss: 0.0251
Epoch 6/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9958 - loss: 0.0239 - val_accuracy: 0.9972 - val_loss: 0.0304
Epoch 7/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9966 - loss: 0.0138 - val_accuracy: 0.9972 - val_loss: 0.0263
Epoch 8/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9977 - loss: 0.0135 - val_accuracy: 0.

In [61]:
# Évaluation sur le jeu de test
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")


42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9978 - loss: 0.0124     
Test Loss: 0.011408409103751183
Test Accuracy: 0.9977628588676453
